<a href="https://colab.research.google.com/github/asheshniraad1-hub/W2052152_CRM-CM_CW1/blob/main/CRM_%26_CM_CW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, io, json, time, hashlib, datetime as dt
import numpy as np
import pandas as pd
import requests
import yfinance as yf
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
CONST_PATH = f"{DATA_DIR}/sp500_constituents.csv"

if CREATE_CONSTITUENT_SNAPSHOT:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text  # fetch page
    tbl = pd.read_html(io.StringIO(html))[0]            # first table = constituents
    tbl["Symbol"] = tbl["Symbol"].str.replace(".", "-", regex=False)  # BRK.B -> BRK-B (Yahoo format)
    tbl["retrieval_date"] = RUN_DATE                    # record when we took the snapshot
    tbl.to_csv(CONST_PATH, index=False)                 # save so results are repeatable

const = pd.read_csv(CONST_PATH)                         # always read from the saved file
CONST_RETRIEVAL = const["retrieval_date"].iloc[0]
tickers = const["Symbol"].tolist()
print(f"{len(tickers)} tickers, list retrieved on {CONST_RETRIEVAL}")
const[['Symbol', 'Security', 'GICS Sector']].head(10)

503 tickers, list retrieved on 2026-11-10


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology
5,ADBE,Adobe Inc.,Information Technology
6,AMD,Advanced Micro Devices,Information Technology
7,AES,AES Corporation,Utilities
8,AFL,Aflac,Financials
9,A,Agilent Technologies,Health Care


In [ ]:
# Study period and validation settings. The end date is exclusive.
START_DATE = "2021-01-01"
END_DATE = "2025-01-01"
BENCHMARK = "^GSPC"          # S&P 500 index
BASE_WINDOW = 20             # sessions used for the volume baseline
ACTIVITY_WINDOW = 60         # sessions used for Frequency and Monetary
HIGH_VOL_MULT = 2.0          # high-volume day = volume > 2 x baseline
MIN_AVAILABILITY = 0.80      # share must have prices on at least 80% of trading days
MIN_WEEKS = 52               # share must have at least 52 weekly returns

In [ ]:
# IMPORTANT: prices are never imputed. Pre-listing dates remain missing.
FILL_PRICES = False
USE_LIVE_DATA = True         # True = download from Yahoo; False = reuse saved CSV
CREATE_CONSTITUENT_SNAPSHOT = True   # True only for the one-time Wikipedia retrieval
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

RUN_DATE = dt.date.today().isoformat()   # date this notebook was run

In [ ]:
PRICE_PATH = f"{DATA_DIR}/daily_prices_raw.csv"

if USE_LIVE_DATA:
    parts = []
    all_syms = tickers + [BENCHMARK]
    for i in range(0, len(all_syms), 50):               # batches of 50 to avoid rate limits
        batch = all_syms[i:i+50]
        d = yf.download(batch, start=START_DATE, end=END_DATE,
                        auto_adjust=False,              # keep BOTH Close and Adj Close
                        group_by="ticker", threads=True, progress=False)
        parts.append(d)
        time.sleep(1)                                   # be polite to the server
    raw = pd.concat(parts, axis=1)                      # columns = (ticker, field)
    raw.to_csv(PRICE_PATH)                              # preserved snapshot
else:
    raw = pd.read_csv(PRICE_PATH, header=[0, 1], index_col=0, parse_dates=True)

raw = raw.sort_index()
print(raw.shape, raw.index.min().date(), raw.index.max().date())

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')
ERROR:yfinance:
1 Failed download:
ERRO

(1005, 3024) 2021-01-04 2024-12-31


In [ ]:


# Extract unique tickers present in the raw columns (excluding the multi-index headers)
ticker_cols = raw.columns.get_level_values(0).unique()

# Identify tickers that have entirely missing Adj Close prices
failed_tickers = []
for ticker in ticker_cols:
    if ticker in raw.columns.levels[0]:
        # If Adj Close is completely null, mark it as failed
        if raw[ticker]['Adj Close'].isnull().all():
            failed_tickers.append(ticker)

print(f"Found {len(failed_tickers)} tickers with completely missing prices: {failed_tickers}")

# Drop these failed tickers from the raw DataFrame
cleaned_raw = raw.drop(columns=failed_tickers, level=0)
print(f"Original shape: {raw.shape}")
print(f"Cleaned shape: {cleaned_raw.shape}")


Found 5 tickers with completely missing prices: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
Original shape: (1005, 3024)
Cleaned shape: (1005, 2994)
